<a href="https://colab.research.google.com/github/Nico-Araujo/FIAP/blob/main/2TIAO/Global-Solution-1/agente_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# 1. Instala o Ollama no Linux do Google Colab
!apt-get install zstd -y
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Instala as bibliotecas de integração do Python
!pip install langchain langchain-ollama langchain-community chromadb pypdf sentence-transformers

# 3. Inicia o servidor do Ollama em segundo plano para o código conseguir conversar com ele
import subprocess
import time
print("Iniciando o servidor Ollama em segundo plano...")
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5) # Aguarda 5 segundos para o servidor inicializar com segurança

# 4. Baixa o Llama 3 diretamente para o ambiente
print("Baixando o Llama 3 (Isso pode levar de 1 a 2 minutos)...")
!ollama pull llama3
print("\n✅ Ollama configurado e Llama 3 pronto")

In [ ]:
!pip install langchain langchain-google-genai chromadb pypdf python-dotenv langchain-community langchain-core

In [ ]:
import os
import json
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def preparar_banco_vetorial_local():
    print("Lendo documentos da NASA e AARO...")
    loader = PyPDFDirectoryLoader("./base_conhecimento")
    docs = loader.load()

    # Corta os PDFs
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    splits = text_splitter.split_documents(docs)

    print(f"Criando embeddings para {len(splits)} blocos localmente...")
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

    # reseta as dimensões do banco
    vectorstore = Chroma.from_documents(
        documents=splits,
        embedding=embeddings,
        collection_name="skyguard_ollama_collection"
    )
    return vectorstore.as_retriever(search_kwargs={"k": 3})

def formatar_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

def analisar_anomalia_com_ollama(retriever, dados_sensores):
    print("Buscando relatórios históricos relevantes nos PDFs...")
    llm = ChatOllama(model="llama3", temperature=0.2)

    system_prompt = (
        "Você é um analista de inteligência aeroespacial experiente, focado em UAPs (Fenômenos Anômalos Não Identificados).\n"
        "Use os trechos dos relatórios oficiais da NASA e AARO fornecidos abaixo para analisar os dados capturados pelos nossos sensores.\n"
        "Emita um laudo técnico estruturado em português informando se o comportamento coincide com registros históricos governamentais.\n\n"
        "Contexto Histórico Oficial:\n{context}"
    )

    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "Dados de telemetria atuais encaminhados pela equipe de campo:\n{input}")
    ])

    print("Cruzando dados e gerando relatório analítico final com Llama 3 via Ollama...")
    dados_texto = json.dumps(dados_sensores, indent=2, ensure_ascii=False)

    rag_chain = (
        {"context": retriever | formatar_docs, "input": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

    response = rag_chain.invoke(dados_texto)
    return response

# ==========================================
# EXECUÇÃO DO TESTE

mock_dados_visao_e_ml = {
    "alerta": "POSITIVO_ANOMALIA",
    "velocidade_estimada": "Mach 4.5",
    "altitude": "25.000 pés",
    "assinatura_termica": "Inexistente (Sem calor de exaustão de motor)",
    "comportamento": "Trajetória em zigue-zague com aceleração instantânea",
    "formato_visao_computacional": "Esférico luminoso / Orb",
    "propulsao_visivel": False
}

try:
    recuperador = preparar_banco_vetorial_local()

    # Dispara a análise para o Llama 3 local
    relatorio_final = analisar_anomalia_com_ollama(recuperador, mock_dados_visao_e_ml)

    print("\n--- 🛸 RELATÓRIO DO AGENTE DE INTELIGÊNCIA SKYGUARD (OLLAMA / LLAMA 3) ---")
    print(relatorio_final)

except Exception as e:
    print(f"\n❌ Erro ao rodar: {e}")
    print("Dica: Certifique-se de que os PDFs estão na pasta 'base_conhecimento' no menu lateral do Colab.")

In [ ]:
import json
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

# ==========================================================
# 1. SIMULANDO A INTEGRAÇÃO COM A Machine Learning + Visão computacional
# ==========================================================

print("Carregando o Cérebro de Machine Learning (Random Forest) da equipe...")
df = pd.read_csv("anomalias_uap.csv") # O arquivo gerado no python
X = df[["velocidade_mach", "mudanca_direcao_graus", "aceleracao_g", "altitude_km", "assinatura_termica", "formato_codigo"]]
y = df["rotulo"]

modelo_ml_dupla = RandomForestClassifier(random_state=42)
modelo_ml_dupla.fit(X, y)

# ==========================================================
# 2. O FLUXO DE DETECÇÃO AO VIVO (PIPELINE SKYGUARD)
# ==========================================================

def pipeline_skyguard_completo(dados_capturados_pelos_sensores):
    print("\n[!] Novo objeto detectado no espaço aéreo.")

    # 2.1 A Visão Computacional extraiu esses números. Vamos passar para o ML prever:
    df_novo_objeto = pd.DataFrame([dados_capturados_pelos_sensores])

    previsao = modelo_ml_dupla.predict(df_novo_objeto)[0]

    if previsao == 0:
        print("✅ ML Classificou como: OBJETO CONVENCIONAL (Avião/Balão/Satélite). Falso Alarme. O céu está seguro.")
        # Retorna o JSON limpo para o ESP32 não apitar
        return {"alerta": "LIMPO"}

    elif previsao == 1:
        print("🚨 ML Classificou como: ANOMALIA / UAP DETECTADO! Iniciando protocolo de segurança...")

        # 2.2 Prepara o JSON para enviar para o seu ESP32 no Wokwi piscar o vermelho
        json_para_esp32 = {
            "alerta": "POSITIVO_ANOMALIA",
            "velocidade_estimada": f"Mach {dados_capturados_pelos_sensores['velocidade_mach']:.1f}",
            "altitude": f"{dados_capturados_pelos_sensores['altitude_km'] * 3280.84:.0f} pes", # Convertendo KM para Pés
            "formato_visao_computacional": "Orb" if dados_capturados_pelos_sensores['formato_codigo'] == 1 else "Tic-Tac"
        }

        # 2.3 Aciona o agente RAG (Ollama) para gerar o relatório descritivo
        print("📡 Enviando telemetria para o Agente RAG analisar cruzamento com dados da NASA...")
        relatorio = analisar_anomalia_com_ollama(recuperador, json_para_esp32)

        print("\n" + "="*60)
        print(" 🛸 RELATÓRIO DO AGENTE DE INTELIGÊNCIA SKYGUARD")
        print("="*60)
        print(relatorio)
        print("="*60)

        print(f"\n[AÇÃO DO OPERADOR]: Copie a linha abaixo e cole no terminal do Wokwi para disparar o alarme físico!")
        print(json.dumps(json_para_esp32))

        return json_para_esp32

# ==========================================================
# 3. TESTANDO O SISTEMA COM DADOS REAIS
# ==========================================================

# Simulação de um objeto que a Visão Computacional acabou de "ver" na câmera
objeto_detectado_agora = {
    "velocidade_mach": 18.5,            # Super rápido
    "mudanca_direcao_graus": 95.0,      # Curva impossível em ângulo reto
    "aceleracao_g": 300.0,              # Aceleração letal
    "altitude_km": 15.0,                # Alta atmosfera
    "assinatura_termica": 0.02,         # Totalmente frio, sem exaustão
    "formato_codigo": 2                 # 2 = Tic-Tac (no código da sua dupla)
}

# Roda o sistema integrando tudo!
resultado_final = pipeline_skyguard_completo(objeto_detectado_agora)

Carregando o Cérebro de Machine Learning (Random Forest) da equipe...

[!] Novo objeto detectado no espaço aéreo.
🚨 ML Classificou como: ANOMALIA / UAP DETECTADO! Iniciando protocolo de segurança...
📡 Enviando telemetria para o Agente RAG analisar cruzamento com dados da NASA...
Buscando relatórios históricos relevantes nos PDFs...
Cruzando dados e gerando relatório analítico final com Llama 3 via Ollama...

 🛸 RELATÓRIO DO AGENTE DE INTELIGÊNCIA SKYGUARD
LAUDO TÉCNICO

ANÁLISE DE DADOS DE TELEMETRIA ATUAIS ENCaminhADOS PELA EQUIPE DE CAMPO

Objeto: Análise de dados de telemetria atuais para verificar se o comportamento do objeto coincide com registros históricos governamentais.

Resumo:

Os dados de telemetria atuais encaminhados pela equipe de campo indicam a presença de uma anomalia não identificada (UAP) em altitude de 49.213 pés e velocidade estimada de Mach 18,5. O formato de visão computacional é classificado como "Tic-Tac", o que sugere um padrão de movimento irregular.

Anális